# Experiment Runner — FL-IDS Full Evaluation

**Project:** Evaluating Federated Learning for Intrusion Detection in Industrial IoT  
**Module:** `src/experiments.py`  

---

## Purpose

This notebook runs the full experiment grid from the thesis proposal and
produces the results for Chapters 4-5. You can run groups individually
(useful for long-running experiments) or all at once.

## Experiment Groups

| Group | Research Question | Configs | Runs (×3 seeds) |
|-------|------------------|---------|------------------|
| EXP1 | Baselines | Centralized + Local K∈{5,10} | 9 |
| EXP2 | Does FL work? | IID, K=5, E=5, R=50 | 3 |
| EXP3 | Effect of E | E∈{1,5,10} | 9 |
| EXP4 | Effect of R | R∈{10,25,50,100} | 12 |
| EXP5 | IID vs Non-IID | IID, skew=0.7, skew=0.9 | 9 |
| EXP6 | Scalability | K=5 vs K=10 | 6 |
| EXP7 | Participation | frac∈{0.3,0.5,0.7,1.0} | 12 |
| | | **Total** | **~60** |

**Note:** Some runs overlap across groups (e.g., EXP2's IID K=5 E=5 R=50 also
appears in EXP3 and EXP5). The runner auto-skips already-completed experiments,
so running all groups in sequence is safe and efficient.

## Runtime Estimate

Each R=50 FL run takes ~4 minutes on CPU. With ~40 unique FL runs:
- CPU: ~3 hours total
- GPU: ~30 minutes total

You can interrupt and resume at any time — completed experiments are skipped.

---
## Step 0 — Setup

In [1]:
import sys, os, json, glob
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../src/'))

from experiments import (
    ensure_partitions, build_experiment_groups, run_experiment_group,
    summarize_group, run_all, SEEDS,
)
from metrics import (
    load_results, aggregate_seeds, print_comparison_table,
    comm_cost_summary, rounds_to_target,
)

# ── Configuration ─────────────────────────────────────────────────
DATA_PATH = "../data/processed/datasense_preprocessed.csv"
PARTITION_DIR = "../data/partitions"
RESULTS_DIR = "../results"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Seeds: {SEEDS}")

Device: cpu
Seeds: [42, 123, 456]


---
## Step 1 — Generate All Partitions

Creates IID and non-IID partition files for all K×seed combinations.
Idempotent — only creates files that don't exist yet.

In [2]:
partition_map = ensure_partitions(DATA_PATH, PARTITION_DIR)

PARTITION GENERATION
  Saved: noniid_07_K5_seed42.json  (5 clients, 30,030 total samples)
  Saved: noniid_09_K5_seed42.json  (5 clients, 30,030 total samples)
  Saved: noniid_07_K5_seed123.json  (5 clients, 30,030 total samples)
  Saved: noniid_09_K5_seed123.json  (5 clients, 30,030 total samples)
  Saved: noniid_07_K5_seed456.json  (5 clients, 30,030 total samples)
  Saved: noniid_09_K5_seed456.json  (5 clients, 30,030 total samples)
  Saved: noniid_07_K10_seed42.json  (10 clients, 30,030 total samples)
  Saved: noniid_09_K10_seed42.json  (10 clients, 30,030 total samples)
  Saved: noniid_07_K10_seed123.json  (10 clients, 30,030 total samples)
  Saved: noniid_09_K10_seed123.json  (10 clients, 30,030 total samples)
  Saved: noniid_07_K10_seed456.json  (10 clients, 30,030 total samples)
  Saved: noniid_09_K10_seed456.json  (10 clients, 30,030 total samples)

Partitions ready: 18/18


---
## Step 2 — Dry Run (Preview)

See what experiments would be run without executing them.
Useful to verify the grid before committing to a long run.

In [3]:
run_all(
    data_path=DATA_PATH,
    partition_dir=PARTITION_DIR,
    results_dir=RESULTS_DIR,
    device=DEVICE,
    dry_run=True,
)

╔════════════════════════════════════════════════════════════════════╗
║  FL-IDS EXPERIMENT PIPELINE                                        ║
║  Evaluating Federated Learning for IIoT Intrusion Detection        ║
╚════════════════════════════════════════════════════════════════════╝

Data:       ../data/processed/datasense_preprocessed.csv
Partitions: ../data/partitions
Results:    ../results
Device:     cpu
Seeds:      [42, 123, 456]
Started:    2026-03-27 21:07:28
PARTITION GENERATION

Partitions ready: 18/18

Experiment groups: ['EXP1', 'EXP2', 'EXP3', 'EXP4', 'EXP5', 'EXP6', 'EXP7']
Total experiments: 60

[DRY RUN] Experiments that would be executed:

  EXP1 (9 experiments):
    [RUN] centralized_seed42
    [RUN] centralized_seed123
    [RUN] centralized_seed456
    [RUN] local_only_K5_iid_seed42
    [RUN] local_only_K5_iid_seed123
    [RUN] local_only_K5_iid_seed456
    [RUN] local_only_K10_iid_seed42
    [RUN] local_only_K10_iid_seed123
    [RUN] local_only_K10_iid_seed456

  EXP

---
## Step 3 — Run Experiments

Run one group at a time (recommended) or all at once.
Uncomment the group you want to run. Already-completed experiments
are automatically skipped.

**Tip:** Run EXP1 (baselines) first since other analyses reference
the centralized F1 as the upper bound.

In [4]:
# Run baselines first (centralized + local-only)
run_all(DATA_PATH, PARTITION_DIR, RESULTS_DIR, DEVICE, groups_to_run=["EXP1"])

╔════════════════════════════════════════════════════════════════════╗
║  FL-IDS EXPERIMENT PIPELINE                                        ║
║  Evaluating Federated Learning for IIoT Intrusion Detection        ║
╚════════════════════════════════════════════════════════════════════╝

Data:       ../data/processed/datasense_preprocessed.csv
Partitions: ../data/partitions
Results:    ../results
Device:     cpu
Seeds:      [42, 123, 456]
Started:    2026-03-27 21:11:08
PARTITION GENERATION

Partitions ready: 18/18

Experiment groups: ['EXP1']
Total experiments: 9

EXPERIMENT GROUP: EXP1 (9 experiments)

[1/9] centralized_seed42
CENTRALIZED BASELINE (upper bound)
Loaded: 30,030 samples, 17 features
Train: 24,024  |  Test: 6,006
Training for 100 epochs (ADAM, lr=0.001, batch=64)
──────────────────────────────────────────────────────────────────────
  [Centralized] Epoch   1/100  loss=1.3608  acc=0.7388  f1=0.6257
  [Centralized] Epoch  10/100  loss=0.5477  acc=0.8467  f1=0.7497
  [Centraliz

In [5]:
# Run IID FL baseline
run_all(DATA_PATH, PARTITION_DIR, RESULTS_DIR, DEVICE, groups_to_run=["EXP2"])

╔════════════════════════════════════════════════════════════════════╗
║  FL-IDS EXPERIMENT PIPELINE                                        ║
║  Evaluating Federated Learning for IIoT Intrusion Detection        ║
╚════════════════════════════════════════════════════════════════════╝

Data:       ../data/processed/datasense_preprocessed.csv
Partitions: ../data/partitions
Results:    ../results
Device:     cpu
Seeds:      [42, 123, 456]
Started:    2026-03-27 21:23:26
PARTITION GENERATION

Partitions ready: 18/18

Experiment groups: ['EXP2']
Total experiments: 3

EXPERIMENT GROUP: EXP2 (3 experiments)

[1/3] fl_iid_K5_E5_R50_seed42
Loaded: 30,030 samples, 5 clients (iid)
Global test set: 6,006 samples
Created 5 clients (samples: [4749, 4812, 4807, 4791, 4865])
Model size: 44,320 bytes (43.3 KB)
Participation: 5/5 clients/round
Starting FedAvg: R=50, E=5, lr=0.01, batch=64
──────────────────────────────────────────────────────────────────────
  Round   1/50  acc=0.6687  f1=0.5393  loss=1.

In [6]:
# Run effect of local epochs
run_all(DATA_PATH, PARTITION_DIR, RESULTS_DIR, DEVICE, groups_to_run=["EXP3"])

╔════════════════════════════════════════════════════════════════════╗
║  FL-IDS EXPERIMENT PIPELINE                                        ║
║  Evaluating Federated Learning for IIoT Intrusion Detection        ║
╚════════════════════════════════════════════════════════════════════╝

Data:       ../data/processed/datasense_preprocessed.csv
Partitions: ../data/partitions
Results:    ../results
Device:     cpu
Seeds:      [42, 123, 456]
Started:    2026-03-27 21:33:25
PARTITION GENERATION

Partitions ready: 18/18

Experiment groups: ['EXP3']
Total experiments: 9

EXPERIMENT GROUP: EXP3 (9 experiments)

[1/9] fl_iid_K5_E1_R50_seed42
Loaded: 30,030 samples, 5 clients (iid)
Global test set: 6,006 samples
Created 5 clients (samples: [4749, 4812, 4807, 4791, 4865])
Model size: 44,320 bytes (43.3 KB)
Participation: 5/5 clients/round
Starting FedAvg: R=50, E=1, lr=0.01, batch=64
──────────────────────────────────────────────────────────────────────
  Round   1/50  acc=0.4925  f1=0.4014  loss=2.

In [7]:
# Run effect of communication rounds
run_all(DATA_PATH, PARTITION_DIR, RESULTS_DIR, DEVICE, groups_to_run=["EXP4"])

╔════════════════════════════════════════════════════════════════════╗
║  FL-IDS EXPERIMENT PIPELINE                                        ║
║  Evaluating Federated Learning for IIoT Intrusion Detection        ║
╚════════════════════════════════════════════════════════════════════╝

Data:       ../data/processed/datasense_preprocessed.csv
Partitions: ../data/partitions
Results:    ../results
Device:     cpu
Seeds:      [42, 123, 456]
Started:    2026-03-27 22:05:27
PARTITION GENERATION

Partitions ready: 18/18

Experiment groups: ['EXP4']
Total experiments: 12

EXPERIMENT GROUP: EXP4 (12 experiments)

[1/12] fl_iid_K5_E5_R10_seed42
Loaded: 30,030 samples, 5 clients (iid)
Global test set: 6,006 samples
Created 5 clients (samples: [4749, 4812, 4807, 4791, 4865])
Model size: 44,320 bytes (43.3 KB)
Participation: 5/5 clients/round
Starting FedAvg: R=10, E=5, lr=0.01, batch=64
──────────────────────────────────────────────────────────────────────
  Round   1/10  acc=0.6687  f1=0.5393  loss

In [8]:
# Run IID vs Non-IID comparison (KEY experiment for thesis)
run_all(DATA_PATH, PARTITION_DIR, RESULTS_DIR, DEVICE, groups_to_run=["EXP5"])

╔════════════════════════════════════════════════════════════════════╗
║  FL-IDS EXPERIMENT PIPELINE                                        ║
║  Evaluating Federated Learning for IIoT Intrusion Detection        ║
╚════════════════════════════════════════════════════════════════════╝

Data:       ../data/processed/datasense_preprocessed.csv
Partitions: ../data/partitions
Results:    ../results
Device:     cpu
Seeds:      [42, 123, 456]
Started:    2026-03-27 23:20:15
PARTITION GENERATION

Partitions ready: 18/18

Experiment groups: ['EXP5']
Total experiments: 9

EXPERIMENT GROUP: EXP5 (9 experiments)

[1/9] fl_iid_K5_E5_R50_seed42
Loaded: 30,030 samples, 5 clients (iid)
Global test set: 6,006 samples
Created 5 clients (samples: [4749, 4812, 4807, 4791, 4865])
Model size: 44,320 bytes (43.3 KB)
Participation: 5/5 clients/round
Starting FedAvg: R=50, E=5, lr=0.01, batch=64
──────────────────────────────────────────────────────────────────────
  Round   1/50  acc=0.6687  f1=0.5393  loss=1.

In [9]:
# Run scalability (K=5 vs K=10)
run_all(DATA_PATH, PARTITION_DIR, RESULTS_DIR, DEVICE, groups_to_run=["EXP6"])

╔════════════════════════════════════════════════════════════════════╗
║  FL-IDS EXPERIMENT PIPELINE                                        ║
║  Evaluating Federated Learning for IIoT Intrusion Detection        ║
╚════════════════════════════════════════════════════════════════════╝

Data:       ../data/processed/datasense_preprocessed.csv
Partitions: ../data/partitions
Results:    ../results
Device:     cpu
Seeds:      [42, 123, 456]
Started:    2026-03-27 23:50:29
PARTITION GENERATION

Partitions ready: 18/18

Experiment groups: ['EXP6']
Total experiments: 6

EXPERIMENT GROUP: EXP6 (6 experiments)

[1/6] fl_iid_K5_E5_R50_seed42
Loaded: 30,030 samples, 5 clients (iid)
Global test set: 6,006 samples
Created 5 clients (samples: [4749, 4812, 4807, 4791, 4865])
Model size: 44,320 bytes (43.3 KB)
Participation: 5/5 clients/round
Starting FedAvg: R=50, E=5, lr=0.01, batch=64
──────────────────────────────────────────────────────────────────────
  Round   1/50  acc=0.6687  f1=0.5393  loss=1.

In [10]:
# Run partial participation
run_all(DATA_PATH, PARTITION_DIR, RESULTS_DIR, DEVICE, groups_to_run=["EXP7"])

╔════════════════════════════════════════════════════════════════════╗
║  FL-IDS EXPERIMENT PIPELINE                                        ║
║  Evaluating Federated Learning for IIoT Intrusion Detection        ║
╚════════════════════════════════════════════════════════════════════╝

Data:       ../data/processed/datasense_preprocessed.csv
Partitions: ../data/partitions
Results:    ../results
Device:     cpu
Seeds:      [42, 123, 456]
Started:    2026-03-28 00:10:42
PARTITION GENERATION

Partitions ready: 18/18

Experiment groups: ['EXP7']
Total experiments: 12

EXPERIMENT GROUP: EXP7 (12 experiments)

[1/12] fl_iid_K5_E5_R50_frac03_seed42
Loaded: 30,030 samples, 5 clients (iid)
Global test set: 6,006 samples
Created 5 clients (samples: [4749, 4812, 4807, 4791, 4865])
Model size: 44,320 bytes (43.3 KB)
Participation: 1/5 clients/round
Starting FedAvg: R=50, E=5, lr=0.01, batch=64
──────────────────────────────────────────────────────────────────────
  Round   1/50  acc=0.6474  f1=0.545

---
## Step 4 — View All Summaries

Load and display aggregated results from every completed group.

In [11]:
for group in ["EXP1", "EXP2", "EXP3", "EXP4", "EXP5", "EXP6", "EXP7"]:
    group_dir = os.path.join(RESULTS_DIR, group)
    if os.path.exists(group_dir) and any(f.endswith('.json') for f in os.listdir(group_dir)):
        print(f"\n{'=' * 70}")
        print(f"  {group}")
        print(f"{'=' * 70}")
        summarize_group(group, RESULTS_DIR, verbose=True)
    else:
        print(f"\n  {group}: not yet run")


  EXP1
  centralized                               F1=0.8646 ± 0.0094  Acc=0.9152 ± 0.0057  (n=3)
  local_only_K10_iid                        F1=N/A  Acc=N/A  (n=3)
  local_only_K5_iid                         F1=N/A  Acc=N/A  (n=3)

  EXP2
  fl_iid_K5_E5_R50                          F1=0.7662 ± 0.0083  Acc=0.8455 ± 0.0028  (n=3)

  EXP3
  fl_iid_K5_E10_R50                         F1=0.7983 ± 0.0115  Acc=0.8686 ± 0.0043  (n=3)
  fl_iid_K5_E1_R50                          F1=0.6569 ± 0.0077  Acc=0.7723 ± 0.0094  (n=3)
  fl_iid_K5_E5_R50                          F1=0.7662 ± 0.0083  Acc=0.8455 ± 0.0028  (n=3)

  EXP4
  fl_iid_K5_E5_R10                          F1=0.6568 ± 0.0052  Acc=0.7716 ± 0.0077  (n=3)
  fl_iid_K5_E5_R100                         F1=0.7930 ± 0.0160  Acc=0.8672 ± 0.0076  (n=3)
  fl_iid_K5_E5_R25                          F1=0.7207 ± 0.0115  Acc=0.8166 ± 0.0026  (n=3)
  fl_iid_K5_E5_R50                          F1=0.7662 ± 0.0083  Acc=0.8455 ± 0.0028  (n=3)

  EXP5
  fl_ii

In [12]:
all_results = {}
for group in ["EXP1", "EXP2", "EXP3", "EXP4", "EXP5", "EXP6", "EXP7"]:
    group_dir = os.path.join("../results", group)
    if not os.path.exists(group_dir):
        continue
    all_results[group] = {}
    for f in sorted(os.listdir(group_dir)):
        if f.endswith(".json"):
            with open(os.path.join(group_dir, f)) as fh:
                data = json.load(fh)
                # Keep only what we need (drop per-round history to save size)
                slim = {
                    "config": data.get("config", {}),
                    "final_metrics": data.get("final_metrics", {}),
                    "best_metrics": data.get("best_metrics", {}),
                    "best_epoch": data.get("best_epoch"),
                    "total_bytes": data.get("total_bytes", 0),
                    "total_time_sec": data.get("total_time_sec", 0),
                    "wall_time_sec": data.get("wall_time_sec", 0),
                    # Keep avg/ensemble for local-only
                    "avg_metrics": data.get("avg_metrics", {}),
                    "ensemble_metrics": data.get("ensemble_metrics", {}),
                    # Keep first/last 3 rounds of history for convergence check
                    "history_first3": data.get("history", [])[:3],
                    "history_last3": data.get("history", [])[-3:],
                    "n_rounds": len(data.get("history", [])),
                }
                all_results[group][f.replace(".json", "")] = slim

with open("../results/all_results_summary.json", "w") as f:
    json.dump(all_results, f, indent=2)

# Print size
size = os.path.getsize("../results/all_results_summary.json") / 1024
print(f"Saved: all_results_summary.json ({size:.0f} KB)")
print(f"Groups: {list(all_results.keys())}")
print(f"Total experiments: {sum(len(v) for v in all_results.values())}")

Saved: all_results_summary.json (288 KB)
Groups: ['EXP1', 'EXP2', 'EXP3', 'EXP4', 'EXP5', 'EXP6', 'EXP7']
Total experiments: 60


---
## Summary

### Experiment grid coverage

| Proposal Objective | Experiment Group | Variable |
|--------------------|-----------------|----------|
| O1: FL feasibility | EXP2 | IID baseline |
| O2: Hyperparameter impact | EXP3 (E), EXP4 (R), EXP6 (K) | E, R, K |
| O3: Non-IID impact | EXP5 | Data heterogeneity |
| O4: Communication efficiency | EXP4 | Rounds-to-target |
| O5: Robustness | EXP7 | Partial participation |

### Output files
- `src/experiments.py` — CLI-runnable experiment pipeline
- `results/EXP{1-7}/*.json` — all individual experiment results

### Next step
→ **Analysis and visualization**: Load results, produce thesis-quality
  plots (convergence curves, bar charts, heatmaps), and write Chapter 4.